In [0]:
%python
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               StringType, DoubleType)

ruta = "/Volumes/workspace/electiva/mi_volumen_datos/"

# Esquemas explícitos. Las fechas se dejan como STRING (formato M/d/yyyy) y se convierten en SQL
esquemas = {
    "customers": StructType([
        StructField("CustomerID", IntegerType(), True),
        StructField("FirstName", StringType(), True),
        StructField("LastName", StringType(), True),
        StructField("FullName", StringType(), True)]),
    "orders": StructType([
        StructField("SalesOrderID", IntegerType(), True),
        StructField("SalesOrderDetailID", IntegerType(), True),
        StructField("OrderDate", StringType(), True),
        StructField("DueDate", StringType(), True),
        StructField("ShipDate", StringType(), True),
        StructField("EmployeeID", IntegerType(), True),
        StructField("CustomerID", IntegerType(), True),
        StructField("SubTotal", DoubleType(), True),
        StructField("TaxAmt", DoubleType(), True),
        StructField("Freight", DoubleType(), True),
        StructField("TotalDue", DoubleType(), True),
        StructField("ProductID", IntegerType(), True),
        StructField("OrderQty", IntegerType(), True),
        StructField("UnitPrice", DoubleType(), True),
        StructField("UnitPriceDiscount", DoubleType(), True),
        StructField("LineTotal", DoubleType(), True)]),
    "products": StructType([
        StructField("ProductID", IntegerType(), True),
        StructField("ProductNumber", StringType(), True),
        StructField("ProductName", StringType(), True),
        StructField("ModelName", StringType(), True),
        StructField("MakeFlag", IntegerType(), True),
        StructField("StandardCost", DoubleType(), True),
        StructField("ListPrice", DoubleType(), True),
        StructField("SubCategoryID", IntegerType(), True)]),
    "product_categories": StructType([
        StructField("CategoryID", IntegerType(), True),
        StructField("CategoryName", StringType(), True)]),
    "product_sub_categories": StructType([
        StructField("SubCategoryID", IntegerType(), True),
        StructField("CategoryID", IntegerType(), True),
        StructField("SubCategoryName", StringType(), True)]),
}

# tabla -> archivo csv
archivos = {
    "customers": "customers.csv",
    "orders": "orders.csv",
    "products": "products.csv",
    "product_categories": "productcategories.csv",
    "product_sub_categories": "productsubcategories.csv",
    "employee": "employees.csv",          # sin esquema explícito (no se usa en el parcial)
    "vendor": "vendors.csv",
    "vendor_product": "vendorproduct.csv",
}

for tabla, archivo in archivos.items():
    lector = spark.read.option("header", "true")
    if tabla in esquemas:
        lector = lector.schema(esquemas[tabla])
    df = lector.csv(ruta + archivo)
    df.write.format("delta").mode("overwrite").saveAsTable(f"workspace.electiva.{tabla}")
    print(tabla, df.count())

In [0]:
%sql
SHOW TABLES IN workspace.electiva;
SELECT * FROM workspace.electiva.orders LIMIT 5;

Punto 1: user_name

In [0]:
%sql
SELECT 
    c.customerid,
    c.fullname,
    CONCAT(c.customerid, '-', UPPER(LEFT(c.firstname, 2)), '-', LOWER(LEFT(c.lastname, 2)), '-AW') AS user_name
FROM workspace.electiva.customers AS c;

Punto 2: tabla page_user

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.electiva.page_user (
    customerid INT,
    user_name STRING
) USING DELTA
COMMENT 'Usuario del sitio web por cliente';

In [0]:
%sql
INSERT INTO workspace.electiva.page_user
SELECT 
    c.customerid,
    CONCAT(c.customerid, '-', UPPER(LEFT(c.firstname, 2)), '-', LOWER(LEFT(c.lastname, 2)), '-AW') AS user_name
FROM workspace.electiva.customers AS c;

In [0]:
%sql
SELECT * FROM workspace.electiva.page_user;

Punto 3: descuentos de agosto 2011

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.electiva.v_util_ordenes_mitad_ano AS
WITH base AS (
    SELECT 
        p.productname,
        p.modelname,
        u.user_name,
        o.salesorderid,
        cat.categoryname,
        to_date(o.orderdate, 'M/d/yyyy') AS orderdate,
        o.linetotal
    FROM workspace.electiva.orders AS o
    JOIN workspace.electiva.page_user AS u                ON o.customerid    = u.customerid
    JOIN workspace.electiva.products AS p                 ON o.productid     = p.productid
    JOIN workspace.electiva.product_sub_categories AS sc  ON p.subcategoryid = sc.subcategoryid
    JOIN workspace.electiva.product_categories AS cat     ON sc.categoryid   = cat.categoryid
    WHERE year(to_date(o.orderdate, 'M/d/yyyy'))  = 2011
      AND month(to_date(o.orderdate, 'M/d/yyyy')) = 8
),
con_descuento AS (
    SELECT 
        *,
        CASE 
            WHEN categoryname ILIKE 'Bikes'       THEN 0.10
            WHEN categoryname ILIKE 'Clothing'    THEN 0.15
            WHEN categoryname ILIKE 'Components'  THEN 0.08
            WHEN categoryname ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`
    FROM base
)
SELECT 
    productname, modelname, user_name, salesorderid, categoryname, orderdate, linetotal,
    `descuento_mitad_año`,
    linetotal - linetotal * `descuento_mitad_año` AS total_descuentos_aplicados
FROM con_descuento;

In [0]:
%sql
SELECT * FROM workspace.electiva.v_util_ordenes_mitad_ano;

Punto 4: vista materializada customer_class

In [0]:
%sql
CREATE MATERIALIZED VIEW workspace.electiva.customer_class AS
SELECT 
    COUNT(DISTINCT o.salesorderid) AS amount_of_orders,
    u.user_name,
    c.fullname AS customerfullname,
    CASE 
        WHEN COUNT(DISTINCT o.salesorderid) >= 3 THEN 'Oro'
        WHEN COUNT(DISTINCT o.salesorderid)  = 2 THEN 'Plata'
        ELSE 'Ocasional'
    END AS user_level,
    o.customerid
FROM workspace.electiva.orders AS o
JOIN workspace.electiva.customers AS c ON o.customerid = c.customerid
JOIN workspace.electiva.page_user AS u ON o.customerid = u.customerid
WHERE year(to_date(o.orderdate, 'M/d/yyyy')) = 2011
GROUP BY o.customerid, u.user_name, c.fullname;

In [0]:
%sql
SELECT * FROM workspace.electiva.customer_class;